# Predição - Classificação - Trecho verdadeiro: Prova de Fogo - Jaqueta

Autor:  Viviane da Rosa Sommer

Atualizado:05/03/2024

## Objetivo:
Notebook para fazer a predição do modelo de Coral-Sol (V3 - Classificação) no trecho que contém Coral-Sol, dos vídeos de Jaqueta.
Essa versão gera um vídeo com os frames originais junto de suas probabilidades.

## Importações Necessárias

In [ ]:
import cv2
from ultralytics import YOLO

## Declaração de Constantes e Modelos

In [ ]:
model = YOLO('../Coral-Classificacao/Pesos/best_weights_v3.pt')
input_video_path = 'Crop/Cortado_SUB8BIQ25-142-OS006000701300-2025-01-09-T-03_37-20-D030.mp4'
output_video_path = 'crop_tp_class.mp4'

## Geração do clipe TP

In [ ]:
cap = cv2.VideoCapture(input_video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    resized_frame = cv2.resize(frame, (224, 224)) 
    rgb_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2RGB)
    results = model.predict(rgb_frame, verbose=False)
    if results:
        top_indices = results[0].probs.top5[:2] 
        top_confidences = results[0].probs.top5conf[:2]

        x_offset = 10
        y_offset = frame_height // 2 - 30
        for i, (index, confidence) in enumerate(zip(top_indices, top_confidences)):
            class_name = results[0].names[index]
            text = f"{class_name}: {confidence:.2f}"
            y_position = y_offset + i * 30
            cv2.putText(frame, text, (x_offset, y_position), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    out.write(frame)
cap.release()
out.release()
print(f"Processamento concluído! O vídeo de saída foi salvo em: {output_video_path}")

In [ ]:
!jupyter nbconvert --to html Analize_TP.ipynb